# llm

> One call shape for each model: `ask(prompt, sp) -> str`.

This module connects the tutor to a model. The tutor needs only one function, `ask(prompt, sp)`, which returns text.
The module also parses JSON replies and gives `Fake`, a model for tests that runs offline.

In [ ]:
#| default_exp llm

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| export
import json, os, re
from fastcore.utils import *

## The model

Each surface gives its own `ask`.

| surface | `ask` |
|---|---|
| web app, notebook | `rishi_ask`: a new rishi `Chat` for each call, on `$ABHYASA_MODEL` (default `claude/sonnet`) |
| ramabana | `agent.oneshot`, so the routing of ramabana applies |
| tests | `Fake` |

In [ ]:
#| export
DFLT_MODEL = 'claude/sonnet'

def rishi_ask(model:str=None):
    "An `ask(prompt, sp='')` on a rishi `Chat` for `model`, else `$ABHYASA_MODEL`, else `DFLT_MODEL`."
    from rishi import Chat, resp_text
    model = model or os.getenv('ABHYASA_MODEL') or DFLT_MODEL
    def ask(prompt, sp=''):
        chat = Chat(model, sp=sp)
        try: return resp_text(chat(prompt))
        finally: chat.close()
    ask.model = model
    return ask

## JSON replies

`parse_json` finds the JSON in a reply. `ask_json` asks again one time when the reply is not valid JSON.

In [ ]:
#| export
def parse_json(s:str):
    "The first JSON object or array in `s`, also inside a ``` fence. Raise `ValueError` when there is none."
    s = s.strip()
    dec = json.JSONDecoder()
    # the whole reply first, then each fence, then the first object or array in the text: a fence inside a JSON string stays whole
    for t in [s, *(m.strip() for m in re.findall(r'```(?:json)?\s*(.*?)```', s, re.S))]:
        try: return json.loads(t)
        except json.JSONDecodeError: pass
    for i, ch in enumerate(s):
        if ch in '{[':
            try: return dec.raw_decode(s, i)[0]
            except json.JSONDecodeError: continue
    raise ValueError(f'no JSON in the reply: {s[:200]!r}')

def ask_json(ask, prompt:str, sp:str='', tries:int=2):
    "Ask for JSON and parse it. On a bad reply, ask one more time with the parse error."
    p = prompt + '\n\nReply with only one JSON value and no other text.'
    for i in range(tries):
        out = ask(p, sp)
        try: return parse_json(out)
        except ValueError as e:
            if i == tries - 1: raise
            p = f'{prompt}\n\nYour last reply was not valid JSON ({e}). Reply with only one JSON value.'

`parse_json` reads JSON in a fence, JSON inside other text, and arrays. A fence inside a JSON string does not break it. It fails when there is no JSON.

In [ ]:
test_eq(parse_json('```json\n{"a": 1}\n```'), {'a': 1})
test_eq(parse_json('Here it is: {"a": [1, 2]} thanks'), {'a': [1, 2]})
test_eq(parse_json('[{"x": 1}]'), [{'x': 1}])
test_fail(lambda: parse_json('nothing'), contains='no JSON')
test_eq(parse_json('{"answer": "Use ```python\\nx = 1\\n``` here."}'), {'answer': 'Use ```python\nx = 1\n``` here.'})
test_eq(parse_json('Sure. {"a": 1} and then {"b": 2}'), {'a': 1})

## A model for tests

`Fake` replies from a dict of rules. The first rule with a key in the prompt gives the reply.
It sends a dict or a list reply as JSON. It keeps each call in `calls`.

In [ ]:
#| export
class Fake:
    "A stand-in model for tests: `Fake({'grade': {...}, 'question': {...}})`."
    def __init__(self, rules:dict=None, dflt='{}'): self.rules, self.dflt, self.calls = rules or {}, dflt, []
    def __call__(self, prompt, sp=''):
        self.calls.append((prompt, sp))
        r = next((v for k, v in self.rules.items() if k in prompt or k in sp), self.dflt)
        if callable(r): r = r(prompt, sp)
        return r if isinstance(r, str) else json.dumps(r)

A matching rule gives its reply, and `Fake` records the call.

In [ ]:
f = Fake({'GRADE': dict(correct=True)})
test_eq(ask_json(f, 'GRADE this'), dict(correct=True))
test_eq(len(f.calls), 1)

After a reply that is not JSON, `ask_json` asks again and gets the JSON.

In [ ]:
bad = Fake(dict(X=lambda p, sp: 'oops' if 'not valid' not in p else '{"ok": 1}'))
test_eq(ask_json(bad, 'X'), {'ok': 1})

## A model for each task

The tutor does five tasks, and each task can use its own model. A quick model can grade and give hints, and a strong model can
write questions and lessons. The Settings page saves the choice, and `Router` reads it on each call, so a change works at once.

| task | what the model does |
|---|---|
| `question` | writes a question and its hidden model answer |
| `grade` | reads the conversation and grades it |
| `hint` | gives the next hint |
| `lesson` | teaches the smallest step after a miss |
| `extract` | finds the concepts in a text |

In [ ]:
#| export
TASKS = ('question', 'grade', 'hint', 'lesson', 'extract')

`model_ask` gives an `ask` for any model name. With ramabana installed (`pip install abhyasa[models]`), the name can be
anything in ramabana's catalogue: Claude Code, Codex, OpenAI, Copilot, Gemini, or a model on this machine (LiteRT, mlx,
llama.cpp, Ollama). ramabana resolves the name and gives a rishi backend. Without ramabana, the name goes to rishi as it is.
One lock for each model makes the calls take turns, because a ramabana backend refuses a second call while one runs.

In [ ]:
#| export
import threading, time
_asks, _asks_lock = {}, threading.Lock()

def _ramabana_ask(name):
    from ramabana.core import resolve
    from ramabana.runtime import make_backend
    b, lock = make_backend(resolve(name)), threading.Lock()
    def ask(prompt, sp=''):
        with lock: out = b.oneshot(prompt, sp)
        if not out and getattr(b, 'note', ''): raise RuntimeError(b.note)
        return out
    return ask

def model_ask(name:str):
    "An `ask(prompt, sp='')` for model `name`, made once and kept."
    with _asks_lock:
        if name not in _asks:
            try: _asks[name] = _ramabana_ask(name)
            except ImportError: _asks[name] = rishi_ask(name)
            _asks[name].model = name
        return _asks[name]

`catalog` lists the models that can run here, from ramabana, for the Settings page. It keeps the list for five minutes, because the look-up asks each provider.

In [ ]:
#| export
FALLBACK = ['claude/sonnet', 'claude/haiku', 'claude/opus']
_cat = {}

def catalog(ttl:float=300.) -> list:
    "The models that can run here, as dicts with `value`, `label`, `provider` and `source`."
    if _cat.get('at', 0) > time.time() - ttl: return _cat['rows']
    try:
        from ramabana.core import available_models
        rows = available_models()
    except Exception: rows = [dict(value=m, label=m, provider='claude', source='rishi') for m in FALLBACK]
    _cat.update(rows=rows, at=time.time())
    return rows

`Router` is the `ask` the app gives the tutor. Called as it is, it uses the default model. `for_(task)` gives the `ask` for one
task: the model the settings name for it, else the default.

In [ ]:
#| export
class Router:
    "The tutor's model: a default `ask`, and one model for each task that the settings name."
    def __init__(self, dflt, models=None):
        self.dflt, self.model = dflt, getattr(dflt, 'model', '')
        self.models = models or (lambda: __import__('abhyasa.voice', fromlist=['settings']).settings().get('models', {}))
    def __call__(self, prompt, sp=''): return self.dflt(prompt, sp)
    def name(self, task:str) -> str:
        "The model for `task`, or '' for the default."
        return (self.models() or {}).get(task) or ''
    def for_(self, task:str):
        "The `ask` for `task`."
        n = self.name(task)
        return model_ask(n) if n else self.dflt

def for_task(ask, task:str):
    "The `ask` for `task` when `ask` routes by task, else `ask` itself."
    return ask.for_(task) if hasattr(ask, 'for_') else ask

A router with no settings uses the default for every task. A task that names a model gets that model.

In [ ]:
dflt = Fake(dflt='default')
r = Router(dflt, models=lambda: {})
test_eq(r('x'), 'default')
test_is(r.for_('grade'), dflt)
named = Fake(dflt='named'); named.model = 'fast'
_asks['fast'] = named
r = Router(dflt, models=lambda: dict(grade='fast'))
test_eq(r.for_('grade')('x'), 'named')
test_eq(r.for_('question')('x'), 'default')
test_is(for_task(dflt, 'grade'), dflt)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()